In [1]:
import pandas as pd
import numpy as np

In [ ]:
players = pd.read_csv(r"C:\Users\HEMANATH\Desktop\FIFA\data\raw\players.csv")
appearances = pd.read_csv(r"C:\Users\HEMANATH\Desktop\FIFA\data\raw\appearances.csv")

In [3]:
print(players.shape)
print(appearances.shape)

players.head()

(47701, 26)
(1885688, 13)


,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,IFM,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000.0,3000000.0


In [5]:
appearances.head()

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45
3,2234418_73333,2234418,73333,1274,76,2012-07-05,Vegar Hedenstad,ELQ,0,0,0,0,90
4,2234421_122011,2234421,122011,195,3008,2012-07-05,Markus Henriksen,ELQ,0,0,0,1,90


In [6]:
player_stats=(appearances.groupby('player_id').agg(
    matches=('game_id','count'),
    goals=('goals','sum'),
    assists=('assists','sum'),
    yellow_cards=('yellow_cards','sum'),
    red_cards=('red_cards','sum'),
    minutes_played=('minutes_played','sum')
)
.reset_index()
)
player_stats.head()

,player_id,matches,goals,assists,yellow_cards,red_cards,minutes_played
0,10,136,48,25,19,0,8808
1,26,152,0,0,4,2,13508
2,65,122,38,13,11,1,8788
3,77,4,0,0,0,0,307
4,80,12,0,0,0,0,1080


In [7]:
master_df=players.merge(player_stats,on='player_id',how='left')
master_df.shape

(47701, 32)

In [8]:
stat_cols = [
    "matches",
    "goals",
    "assists",
    "yellow_cards",
    "red_cards",
    "minutes_played"
]

master_df[stat_cols] = master_df[stat_cols].fillna(0)

In [9]:
master_df['data_of_birth']=pd.to_datetime(master_df['date_of_birth'],errors='coerce')

today=pd.Timestamp.today()

master_df['age']=((today-master_df['data_of_birth']).dt.days/365.25)

master_df['age']=master_df['age'].fillna(master_df['age'].median())

master_df['age']=master_df['age'].astype('int64')

In [10]:
master_df=master_df[master_df['minutes_played']>=900]

print(master_df.shape)

(17467, 34)


In [14]:
master_df["goals_per90"] = (
    master_df["goals"]
    /
    (master_df["minutes_played"] / 90)
)

master_df["goals_per90"] = (
    master_df["goals_per90"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

In [ ]:
master_df["assists_per90"] = (
    master_df["assists"]
    /
    (master_df["minutes_played"] / 90)
)

master_df["assists_per90"] = (
    master_df["assists_per90"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,goals,assists,yellow_cards,red_cards,minutes_played,data_of_birth,age,goal_per90,assists_per90,goals_per90
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,48.0,25.0,19.0,0.0,8808.0,1978-06-09,48,48.0,0.255450,0.490463
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,0.0,0.0,4.0,2.0,13508.0,1980-08-06,45,0.0,0.000000,0.000000
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,38.0,13.0,11.0,1.0,8788.0,1981-01-30,45,38.0,0.133136,0.389167
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,0.0,0.0,0.0,0.0,1080.0,1981-03-18,45,0.0,0.000000,0.000000
5,109,NaN,Dedê,Dedê,2013,825,dede,Brazil,Belo Horizonte,Brazil,...,1.0,2.0,4.0,0.0,3584.0,1978-04-18,48,1.0,0.050223,0.025112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46980,1360186,Barney,Stewart,Barney Stewart,2025,1191,barney-stewart,England,London,Scotland,...,10.0,2.0,1.0,0.0,1430.0,2004-04-07,22,10.0,0.125874,0.629371
47016,1364454,Christian,Kofane,Christian Kofane,2025,15,christian-kofane,Cameroon,Douala,Cameroon,...,7.0,9.0,5.0,0.0,1983.0,2006-07-26,19,7.0,0.408472,0.317700
47212,1380311,Prince,Amoako Junior,Prince Amoako Junior,2025,2778,prince-amoako-junior,Ghana,Amasaman,Ghana,...,7.0,5.0,1.0,0.0,1896.0,2007-02-19,19,7.0,0.237342,0.332278
47280,1390649,Yan,Diomande,Yan Diomande,2025,23826,yan-diomande,Cote d'Ivoire,Abidjan,Cote d'Ivoire,...,16.0,11.0,4.0,0.0,3618.0,2006-11-14,19,16.0,0.273632,0.398010


In [33]:
master_df['goal_contribution_per90']=(
    master_df['goals_per90']+master_df['assists_per90']
)

In [34]:
master_df["cards_per90"] = (
    (
        master_df["yellow_cards"]
        +
        master_df["red_cards"]
    )
    /
    (master_df["minutes_played"] / 90)
)

master_df["cards_per90"] = (
    master_df["cards_per90"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

In [35]:
master_df['log_market_value']=np.log1p(master_df['market_value_in_eur'])

master_df['log_market_value']

0        13.815512
1        13.527830
2        13.815512
4        11.512935
5        12.899222
           ...    
46980          NaN
47016          NaN
47212          NaN
47280    17.622173
47301    13.815512
Name: log_market_value, Length: 17467, dtype: float64

In [43]:
master_df["intl_goal_ratio"] = (
    master_df["international_goals"]
    /
    (master_df["international_caps"] + 1)
)

In [46]:
master_df['position'].value_counts()

position
Defender      6196
Midfield      5158
Attack        4651
Goalkeeper    1444
Missing         18
Name: count, dtype: int64

In [47]:
def simplify_position(pos):
    if pos=='Attack':
        return 'Attack'
    elif pos=='Midfield':
        return 'Midfield'
    elif pos=='Defender':
        return 'Defender'
    elif pos=='Goalkeeper':
        return 'Goalkeeper'
    else:
        return 'others'


master_df['position_group']=master_df['position'].apply(simplify_position)


In [48]:
master_df['position_group'].value_counts()

position_group
Defender      6196
Midfield      5158
Attack        4651
Goalkeeper    1444
others          18
Name: count, dtype: int64

In [49]:
master_df[
[
    "name",
    "position_group",
    "age",
    "matches",
    "goals",
    "assists",
    "minutes_played",
    "market_value_in_eur",
    "goals_per90",
    "assists_per90",
    "goal_contribution_per90"
]
].head()

,name,position_group,age,matches,goals,assists,minutes_played,market_value_in_eur,goals_per90,assists_per90,goal_contribution_per90
0,Miroslav Klose,Attack,48,136.0,48.0,25.0,8808.0,1000000.0,0.490463,0.255450,0.745913
1,Roman Weidenfeller,Goalkeeper,45,152.0,0.0,0.0,13508.0,750000.0,0.000000,0.000000,0.000000
2,Dimitar Berbatov,Attack,45,122.0,38.0,13.0,8788.0,1000000.0,0.389167,0.133136,0.522303
4,Tom Starke,Goalkeeper,45,12.0,0.0,0.0,1080.0,100000.0,0.000000,0.000000,0.000000
5,Dedê,Defender,48,41.0,1.0,2.0,3584.0,400000.0,0.025112,0.050223,0.075335


In [50]:
master_df.to_csv("data/processed/master_player_features.csv",index=False)

print('file saved')

file saved
